In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Merging RAW_recipes.csv with RAW_recipes_w_search_terms.csv

In [ ]:
df_recipes = pd.read_csv("datasets/RAW_recipes.csv")
df_recipes_w_search_terms = pd.read_csv("datasets/recipes_w_search_terms.csv")

In [ ]:
# Merge the original RAW_recipes dataset and recipes_w_search_terms dataset
new_columns = [col for col in df_recipes_w_search_terms.columns if col not in df_recipes.columns or col == 'id']
df_recipes = pd.merge(df_recipes, df_recipes_w_search_terms[new_columns], on='id', how='inner')

print("Recipes Dataset Info:\n")
df_recipes.info()
print("")
df_recipes.head()

In [ ]:
# Drop the data row with missing name (since there is only 1 such row)
df_recipes = df_recipes.dropna(subset=['name'])
print("Recipes Dataset Info:\n")
df_recipes.info()

In [ ]:
# For data rows with missing description, we concatenate their recipe name and ingredients using '{name} is made using {ingredients}'
df_recipes['description'] = df_recipes.apply(
    lambda row: f"{row['name']} is made using {', '.join(eval(row['ingredients']))}" if pd.isnull(row['description']) else row['description'],
    axis=1
)

print("Recipes Dataset Info:\n")
df_recipes.info()
print("")
df_recipes.head()

In [ ]:
# Convert 'submitted' to datetime
df_recipes['submitted'] = pd.to_datetime(df_recipes['submitted'])


In [ ]:
# Take out Nutrition List Individual Values
df_recipes[['calories', 'total fat (PDV)', 'sugar (PDV)', 'sodium (PDV)', 'protein (PDV)',
    'saturated fat (PDV)', 'carbohydrates (PDV)']] = df_recipes.nutrition.str.split(",", expand=True)

df_recipes['calories'] = df_recipes['calories'].str.replace('[', '')
df_recipes['carbohydrates (PDV)'] = df_recipes['carbohydrates (PDV)'].str.replace(']', '')
df_recipes[['calories', 'total fat (PDV)', 'sugar (PDV)', 'sodium (PDV)', 'protein (PDV)', 'saturated fat (PDV)',
    'carbohydrates (PDV)']] = df_recipes[['calories', 'total fat (PDV)', 'sugar (PDV)', 'sodium (PDV)',
                                  'protein (PDV)', 'saturated fat (PDV)', 'carbohydrates (PDV)']].astype('float')

In [ ]:
# Number of Rows with zero minutes
zero_minutes_count = df_recipes[df_recipes['minutes'] == 0].shape[0]
print(zero_minutes_count)

In [ ]:
# Drop rows where 'minutes' is 0
df_recipes = df_recipes[df_recipes['minutes'] != 0]
df_recipes.info()

In [ ]:
# Number of Rows with zero n_steps
zero_n_steps_count = df_recipes[df_recipes['n_steps'] == 0].shape[0]
print(zero_n_steps_count)

In [ ]:
# Drop rows where 'n_steps' is 0
df_recipes = df_recipes[df_recipes['n_steps'] != 0]
df_recipes.info()

In [ ]:
# Number of Rows with zero n_ingredients
zero_n_ingredients_count = df_recipes[df_recipes['n_ingredients'] == 0].shape[0]
print(zero_n_ingredients_count) # 0 so don't need to drop any rows

In [ ]:
# Number of Rows with zero servings
zero_servings_count = df_recipes[df_recipes['servings'] == 0].shape[0]
print(zero_servings_count) # 0 so don't need to drop any rows

Observed that the data for `serving_size` is formatted by '1 (xxxx g)' for each data row. As such, I will be extracting only the serving size in grams and create a new column `serving_size_grams`.

In [ ]:
# Extract the serving_size grams number into new column 'serving_size_grams'
df_recipes['serving_size_grams'] = df_recipes['serving_size'].str.extract(r'\((\d+)\s*g\)').astype(float)


In [ ]:
# Check for any null values for serving_size_grams
null_serving_size_rows = df_recipes[df_recipes['serving_size_grams'].isnull()]
null_serving_size_rows # only 1 row

In [ ]:
# Find out the serving_size number for the data row containing the null value
null_serving_size_rows['serving_size']

In [ ]:
# Fill with the found value
df_recipes['serving_size_grams'].fillna(475.0, inplace=True)

In [ ]:
# Number of Rows with zero serving_size_grams
zero_serving_size_grams_count = df_recipes[df_recipes['serving_size_grams'] == 0].shape[0]
print(zero_serving_size_grams_count)

In [ ]:
# Drop rows where 'serving_size_grams' is 0
df_recipes = df_recipes[df_recipes['serving_size_grams'] != 0]
df_recipes.info()

In [ ]:
# Drop not needed columns
df_recipes = df_recipes.drop(columns=['serving_size','nutrition'])
df_recipes.info()

In [ ]:
# Save to data folder
df_recipes.to_csv("datasets/df_recipes_merged.csv", index=False)

## Merge df_interactions_with_ratings.csv and df_recipes_merged.csv




In [ ]:
df_recipes_merged = pd.read_csv("datasets/df_recipes_merged.csv")

In [ ]:
df_recipes_merged.head()
df_recipes_merged.info()

In [ ]:
df_interactions = pd.read_csv("datasets/df_interactions_with_ratings.csv")

In [ ]:
df_interactions.head()

In [ ]:
df_aggregated = df_interactions.merge(df_recipes_merged, left_on="recipe_id", right_on="id", how='left')
df_aggregated.head()

In [ ]:
df_aggregated.drop(columns=["id"], inplace=True)

In [ ]:
df_aggregated.info()

In [ ]:
# Some user interaction rows do not have recipes merged to them
unmapped_recipe_ids = set(df_interactions['recipe_id']) - set(df_recipes_merged['id'])
print(len(unmapped_recipe_ids))
unmapped_recipe_ids_list = list(unmapped_recipe_ids)
print(unmapped_recipe_ids_list[:50])

In [ ]:
# Drop the rows
df_aggregated_cleaned = df_aggregated.dropna(subset=df_aggregated.columns[6:])
df_aggregated_cleaned.info()

In [ ]:
# Convert 'date' to datetime
df_aggregated_cleaned['date'] = pd.to_datetime(df_aggregated_cleaned['date'])

In [ ]:
# Save to data folder
df_aggregated_cleaned.to_csv("datasets/df_agg_cleaned.csv", index=False)

In [ ]:
df_aggregated_cleaned.head()